# CS3807 Deep Learning Laboratory: Attention Mechanisms in ResNet50
## Experiments 8, 9: Effect of Attention Type and Position
### Complete Systematic Notebook with Checklist

**Learning Outcome**: Understand how attention mechanisms modify pretrained CNNs and how position influences performance.

---

## SECTION 1: ENVIRONMENT SETUP AND IMPORTS

In [ ]:
# Cell 1: Import all required libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import time
import pickle
import warnings
warnings.filterwarnings('ignore')

# TensorFlow & Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing import image
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.metrics import Accuracy, Precision, Recall, AUC
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_fscore_support

# Visualization
import cv2
from PIL import Image

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))
print("All imports successful!")

In [ ]:
# Cell 2: Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
os.environ['PYTHONHASHSEED'] = '42'

print("Random seeds set for reproducibility")
print(f"NumPy seed: 42")
print(f"TensorFlow seed: 42")

In [ ]:
# Cell 3: Configure visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("Visualization settings configured")

In [ ]:
# Cell 4: Create directories for saving results
os.makedirs('results', exist_ok=True)
os.makedirs('results/models', exist_ok=True)
os.makedirs('results/visualizations', exist_ok=True)
os.makedirs('results/attention_maps', exist_ok=True)
os.makedirs('results/gradcam', exist_ok=True)
os.makedirs('results/metrics', exist_ok=True)

print("Directory structure created:")
print("- results/")
print("  - models/")
print("  - visualizations/")
print("  - attention_maps/")
print("  - gradcam/")
print("  - metrics/")

## SECTION 2: DATASET LOADING AND EXPLORATION

In [ ]:
# Cell 5: Download and load TensorFlow Flowers dataset
import tensorflow_datasets as tfds

# Load dataset info
dataset_info = tfds.builder('tf_flowers')
print(f"Dataset: {dataset_info.name}")
print(f"Homepage: {dataset_info.info.homepage}")
print(f"Description: {dataset_info.info.description[:200]}...")

# Load the full dataset
dataset = tfds.load('tf_flowers', split='train')
print(f"\nDataset loaded successfully")

In [ ]:
# Cell 6: Extract images and labels from dataset
images = []
labels = []

# Class names mapping
class_names = ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']
num_classes = len(class_names)

# Extract all data
for sample in tqdm(dataset, desc='Loading dataset'):
    img = sample['image'].numpy()
    label = sample['label'].numpy()
    
    images.append(img)
    labels.append(label)

images = np.array(images)
labels = np.array(labels)

print(f"Dataset shape: {images.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Number of classes: {num_classes}")
print(f"Class names: {class_names}")

# Count samples per class
unique, counts = np.unique(labels, return_counts=True)
print(f"\nSamples per class:")
for cls, count in zip(unique, counts):
    print(f"  {class_names[cls]}: {count}")

In [ ]:
# Cell 7: Visualize sample images from all 5 classes
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
fig.suptitle('Sample Images from Each Class (Required Visualization #1)', fontsize=14, fontweight='bold')

# Get one sample per class
for class_idx in range(num_classes):
    class_indices = np.where(labels == class_idx)[0]
    sample_idx = class_indices[0]
    
    axes[class_idx].imshow(images[sample_idx].astype('uint8'))
    axes[class_idx].set_title(class_names[class_idx].capitalize())
    axes[class_idx].axis('off')

plt.tight_layout()
plt.savefig('results/visualizations/01_sample_images.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Required Visualization #1: Sample images saved")

## SECTION 3: DATA PREPROCESSING AND SPLITTING

In [ ]:
# Cell 8: Resize all images to 224x224x3 (ResNet50 input size)
target_size = (224, 224)
resized_images = []

print(f"Resizing {len(images)} images to {target_size}...")
for img in tqdm(images, desc='Resizing images'):
    # Ensure image is uint8
    if img.dtype != np.uint8:
        img = (img * 255).astype(np.uint8) if img.max() <= 1 else img.astype(np.uint8)
    
    # Resize using PIL
    pil_img = Image.fromarray(img)
    resized_pil = pil_img.resize(target_size, Image.Resampling.LANCZOS)
    resized_images.append(np.array(resized_pil))

images = np.array(resized_images)
print(f"All images resized. Final shape: {images.shape}")
print(f"Image dtype: {images.dtype}")
print(f"Image value range: [{images.min()}, {images.max()}]")

In [ ]:
# Cell 9: Apply ImageNet preprocessing (ResNet50 requirement)
from tensorflow.keras.applications.resnet50 import preprocess_input

print("Applying ImageNet preprocessing for ResNet50...")
images_processed = preprocess_input(images.copy())  # This converts to [-1, 1] range

print(f"Processed images shape: {images_processed.shape}")
print(f"Processed images dtype: {images_processed.dtype}")
print(f"Processed images range: [{images_processed.min():.2f}, {images_processed.max():.2f}]")
print("✓ ImageNet preprocessing applied")

In [ ]:
# Cell 10: Convert labels to one-hot encoding
labels_onehot = keras.utils.to_categorical(labels, num_classes)

print(f"Labels converted to one-hot encoding")
print(f"One-hot labels shape: {labels_onehot.shape}")
print(f"Sample one-hot label (class 0): {labels_onehot[0]}")

In [ ]:
# Cell 11: Perform controlled 70-15-15 train-val-test split
from sklearn.model_selection import train_test_split

# First split: 70% train, 30% temp (val+test)
X_train, X_temp, y_train, y_temp = train_test_split(
    images_processed, labels_onehot, test_size=0.30, random_state=42, stratify=labels
)

# Second split: Split temp 50-50 for validation and test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp.argmax(axis=1)
)

print(f"Dataset Split (70-15-15):")
print(f"  Training: {len(X_train)} samples ({len(X_train)/len(images_processed)*100:.1f}%)")
print(f"  Validation: {len(X_val)} samples ({len(X_val)/len(images_processed)*100:.1f}%)")
print(f"  Testing: {len(X_test)} samples ({len(X_test)/len(images_processed)*100:.1f}%)")

# Verify class distribution in splits
for split_name, y_split in [('Training', y_train), ('Validation', y_val), ('Testing', y_test)]:
    print(f"\n{split_name} class distribution:")
    split_labels = y_split.argmax(axis=1)
    for cls_idx in range(num_classes):
        count = np.sum(split_labels == cls_idx)
        print(f"  {class_names[cls_idx]}: {count}")

In [ ]:
# Cell 12: Define and apply data augmentation (SAME for ALL models)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Create augmentation pipeline
train_augment = ImageDataGenerator(
    horizontal_flip=True,           # Random horizontal flipping
    rotation_range=15,              # Small rotations (±15°)
    zoom_range=0.2,                 # Random zoom (0.8-1.2x)
    width_shift_range=0.1,          # Random translation (10%)
    height_shift_range=0.1,         # Random translation (10%)
    brightness_range=[0.9, 1.1],    # Mild contrast variation (±10%)
    fill_mode='nearest'
)

print("Data augmentation pipeline configured:")
print(" done - Random horizontal flipping")
print(" done - Small rotations (±15°)")
print(" done - Random zoom (0.8-1.2x)")
print(" done - Random translation (±10%)")
print(" done - Mild contrast variation (±10%)")
print("\nIMPORTANT: This same augmentation will be applied to ALL model experiments")

In [ ]:
# Cell 13: Save dataset splits for reproducibility
splits_info = {
    'X_train': X_train,
    'y_train': y_train,
    'X_val': X_val,
    'y_val': y_val,
    'X_test': X_test,
    'y_test': y_test,
    'class_names': class_names,
    'num_classes': num_classes,
    'train_augment': train_augment
}

with open('results/data_splits.pkl', 'wb') as f:
    pickle.dump(splits_info, f)

print("✓ Dataset splits saved to 'results/data_splits.pkl'")
print(f"Total samples: {len(X_train) + len(X_val) + len(X_test)}")

## SECTION 4: RESNET50 ARCHITECTURE ANALYSIS

In [ ]:
# Cell 14: Load pretrained ResNet50 and analyze architecture
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

print("ResNet50 Architecture Summary:")
print("="*60)
base_model.summary()

In [ ]:
# Cell 15: Document ResNet50 feature hierarchy and positions
# Create a reference table for ResNet50 stages
resnet_stages = pd.DataFrame({
    'Stage': ['Stem + MaxPool', 'conv2_x', 'conv3_x', 'conv4_x', 'conv5_x'],
    'Position': ['Before P1', 'P1', 'P2', 'P3', 'P4'],
    'Spatial Resolution': ['56×56', '56×56', '28×28', '14×14', '7×7'],
    'Channels': ['64', '256', '512', '1024', '2048'],
    'Feature Type': ['Low-level', 'Low-level', 'Intermediate', 'High-level', 'High-level'],
    'Content': ['Edges, colors, textures', 'Edges, textures', 'Flower parts, patterns', 'Object structures', 'Class-discriminative info']
})

print("ResNet50 Feature Hierarchy & Attention Positions:")
print("="*120)
print(resnet_stages.to_string(index=False))

# Save reference table
resnet_stages.to_csv('results/resnet_stages_reference.csv', index=False)
print("\n✓ ResNet50 reference table saved")

In [ ]:
# Cell 16: Identify key layers for attention insertion
# Map layer names to positions
layer_map = {
    'P2': 'conv3_block4_out',  # After conv3_x
    'P3': 'conv4_block6_out',  # After conv4_x
    'P4': 'conv5_block3_out'   # After conv5_x
}

print("Attention Insertion Point Mapping:")
print("="*60)
for position, layer_name in layer_map.items():
    try:
        layer = base_model.get_layer(layer_name)
        print(f"\n{position}: {layer_name}")
        print(f"  Output shape: {layer.output_shape}")
    except:
        print(f"\n{position}: {layer_name} - Layer not found")

print("\nThese layers will be used as attention insertion points.")

---

**This is Part 1 of the notebook. Continue to Part 2 for attention mechanism implementations.**

**Checklist Progress**:
- [x] Environment setup
- [x] Dataset loading (Section 2)
- [x] Image preprocessing (Cell 8-9)
- [x] Dataset splitting (Cell 11)
- [x] Data augmentation (Cell 12)
- [x] ResNet50 analysis (Cell 14-16)
- [ ] Attention implementation
- [ ] Model training
- [ ] Evaluation and metrics
- [ ] Visualizations and Grad-CAM